In [ ]:
import akida
import pickle, os

import pandas as pd
import numpy as np

from dataset_NewEEG import filter_rawEEG

used_channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']

t_start = 0
t_end = 250

df = pd.read_csv("sample_eeg_data.csv", delimiter=',')
df = df.loc[:, used_channels]
X = np.array(df)[t_start:t_end].transpose(1,0)
                    
X = filter_rawEEG(X, 0.5, 35)



print(X.shape)


print("================================= \n\n Using Akida Neuromorphic Mode ...  \n")

chip = akida.devices()[0]
#chip = akida.AKD1000()


print(f"=> Akida chip detected = {chip} \n")


akida_model = akida.Model(os.path.join('saved_models', 'LR.fbz'))
with open(os.path.join('saved_models', 'cspLR.pkl'), 'rb') as f:
    csp = pickle.load(f)

X = np.expand_dims(X, 0)
X = csp.transform(X)
X = np.expand_dims(X, 3)


X = ((X - X.min()) / (X.max() - X.min()) * 255).astype(np.uint8)
X = np.pad(X, ((0, 0), (2, 2), (0, 0), (0, 0)), mode='constant', constant_values=0)

#chip.soc.power_measurement_enabled = True

akida_model.map(chip, hw_only=True)

akida_model.predict(X)

#akida_model.map(chip, hw_only = True)



(13, 250)

 Using Akida Neuromorphic Mode ...  

=> Akida chip detected = <akida.core.Device object at 0x0000026253DF0FF0> 



RuntimeError: An actual hardware device is required.